# Cleaning 2.3 - Clean energy calculations

This notebook does the following:
    - Creates variables so that the dataset can be merged with the equipment calculators

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from fill_missing_mode import fill_with_equipment_mode
from assign_set_temp import assign_set_temp

In [2]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_2.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Load labs data
labs = pd.read_csv(
    config.PROCESSED_DATA / "individual_processed_5.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [4]:
# Load equipment and fc calculations
equipment_calculations = pd.read_csv(
    config.SPARK_DATA / "2_Clean" / "equipment_calculations.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

fc_calculations = pd.read_csv(
    config.SPARK_DATA / "2_Clean" / "fc_calculations.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [5]:
# Load merge overrides file
merge_overrides = pd.read_excel(config.CLEANING_WORKBOOKS / "merge_overrides.xlsx")

In [6]:
# Load data dictionaries
equipment_data_dict = pd.read_excel(config.DATA_DICTIONARIES / "data_dictionary.xlsx", sheet_name="Equipment")

In [7]:
# Merge equipment with labs to get institute_id
equipment = equipment.merge(labs[["labgroupid", "institute_id"]], left_on = "labgroupid", right_on = "labgroupid", how = "left")

## (1) Prepare for merging with equipment calculators

### (1a) Generic variables (number, hours, days, door_openings)

In [8]:
# Clean number variable

# If number is missing, set number to 1
equipment.loc[equipment["number"].isna(), "number"] = 1

# Make number integer
equipment["number"] = equipment["number"].astype(int)


In [9]:
# Clean hours and days variables
print(equipment["equipment"].unique())

# Add mask for equipment that has days or hours
has_days_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("fc", "heater", "bath", "cryostat", "microbio", "glassware", "it")))
has_hours_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("heater", "bath", "cryostat", "microbio", "glassware", "it")))
#always_on = ("fridge", "freezer", "ult") #Included for completeness

# If hours is missing or "Unknown", set hours to 9
equipment.loc[(has_hours_mask & (equipment["hours"].isna()) | (equipment["hours"] == "Unknown")), "hours"] = 9

# If days is missing or "Unknown", set days to 255
equipment.loc[(has_days_mask & (equipment["days"].isna()) | (equipment["days"] == "Unknown")), "days"] = 255

# Make hours and days numeric
equipment["hours"] = pd.to_numeric(equipment["hours"], errors="raise")
equipment["days"] = pd.to_numeric(equipment["days"], errors="raise")

['heater' 'it' 'bath' 'fc' 'freezer' 'fridge' 'microbio' 'ult' 'cryostat'
 'glassware' 'incubator']


In [10]:
# Clean door openings (note: door opening is now in minutes!, and this impacts fridge, freezer, and ults)

# Mask for equipment with door openings (fridge, freezer, ult)
has_door_opening_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("freezer", "fridge", "ult")))

# Check unique values for door openings
print("Unique values in door_openings before cleaning:")
print(equipment.loc[has_door_opening_mask, "door_openings"].unique())

# If door openings is unknown or NaN, set to 0
equipment.loc[
    (has_door_opening_mask & 
     (equipment["door_openings"].isna() | equipment["door_openings"].isin(["Unknown"]))), "door_openings"
] = 0

# Make door openings numeric
equipment["door_openings"] = pd.to_numeric(equipment["door_openings"], errors="raise")

Unique values in door_openings before cleaning:
['0' '1' '2' '0.03' '5' '10' '15' '20' '3' '6' '30' '12' 'Unknown' '7' '4'
 '100' '0.3' '0.01' '7.5' '40' '60' '27' '0.1' '0.8' '8' '25' '0.5' '10.0'
 '0.05' '0.06' '2.5' '45' '3.0' '0.0' '42' '1.5']


### (1b) FC variables (hours_open, controller_type, sash_width, face_velocity, lifted, surface)

In [11]:
# Hours open

# Check unique values in hours_open
print("Unique values in hours_open before cleaning:")
print(equipment[equipment["equipment"] == "fc"]["hours_open"].unique())

# Fill missing/unknown values for hours_open with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="hours_open",
    equipment_value="fc",
)

# Make hours_open numeric
equipment["hours_open"] = pd.to_numeric(equipment["hours_open"], errors="raise")

Unique values in hours_open before cleaning:
[ 1.  2.  3. nan  8.  6.  4.  5.  0. 24.]


In [12]:
# Controller type

# Check unique values and distribution of controller_type
print(equipment[equipment["equipment"] == "fc"]["controller_type"].unique())
print(equipment[equipment["equipment"] == "fc"]["controller_type"].value_counts(dropna=False))

# Clean to shortened names ("VAV"/"CAV" instead of "Variable Air Volume (VAV)"/"Constant Air Volume (CAV)")
equipment["controller_type"] = equipment["controller_type"].replace({
    "Variable Air Volume (VAV)": "VAV",
    "Constant Air Volume (CAV)": "CAV"
})

# Fill missing/unknown values for controller_type with "VAV"
equipment.loc[
    (equipment["equipment"] == "fc") & 
    (equipment["controller_type"].isna() | equipment["controller_type"].isin(["Unknown"])),
    "controller_type"
] = "VAV"

print(equipment[equipment["equipment"] == "fc"]["controller_type"].unique())


['Variable Air Volume (VAV)' 'Constant Air Volume (CAV)' nan 'Unknown']
controller_type
Variable Air Volume (VAV)    108
Constant Air Volume (CAV)     64
NaN                           10
Unknown                       10
Name: count, dtype: int64
['VAV' 'CAV']


In [13]:
# Sash width (convert width to meters and numeric)

# Check unique values for sash_width
print(equipment[equipment["equipment"] == "fc"]["sash_width"].unique())

# Set sash_width to the mode within each institute, if missing or unknown
equipment = fill_with_equipment_mode(
    equipment,
    value_col="sash_width",
    equipment_value="fc",
    groupby_cols="institute_id"
)

# If missing or unknown, set to 1500mm
# equipment.loc[
#     (equipment["equipment"] == "fc") & 
#     (equipment["sash_width"].isna() | equipment["sash_width"].isin(["Unknown"])),
#     "sash_width"
# ] = 1500

# Set to numeric and convert from mm to m
equipment["sash_width"] = pd.to_numeric(equipment["sash_width"], errors="raise")/1000

# Check that all values are within 0.5-3m
assert equipment.loc[equipment["equipment"] == "fc", "sash_width"].between(0.5, 3).all(), "Some sash_width values are outside the range of0.5-3 meters."

['1700' '1080' '1100' '1200' '1500' nan 'Unknown' '1200.0' '1070' '1350'
 '1050' '1150' '1450' '1000' '900' '1170' '1110' '2000' '1300' '1250'
 '1190' '1160' '500' '1120' '1900']


In [14]:
# Face velocity

# Print unique values and distribution of face_velocity
print(equipment[equipment["equipment"] == "fc"]["face_velocity"].unique())

# Create indicators for being in m3/h or m/s
equipment.loc[
    (equipment["equipment"] == "fc") &
    (equipment["face_velocity"].str.contains("m3/h", na=False)),
    "face_velocity_unit_m3/h"
] = True
equipment.loc[
    (equipment["equipment"] == "fc") &
    (equipment["face_velocity"].str.contains("m/s", na=False)),
    "face_velocity_unit_m/s"
] = True

# Create variable which is in m3/h (minus " m3/h" from the string and convert to numeric)
equipment.loc[
    (equipment["equipment"] == "fc") & 
    (equipment["face_velocity_unit_m3/h"] == True),
    "face_velocity_m3/h"
] = equipment["face_velocity"].str.replace("m3/h", "")
equipment["face_velocity_m3/h"] = pd.to_numeric(equipment["face_velocity_m3/h"], errors="raise")

# Create variable which is in m/s (minus " m/s" from the string and convert to numeric)
equipment.loc[
    (equipment["equipment"] == "fc") &
    (equipment["face_velocity_unit_m/s"] == True),
    "face_velocity_m/s"
] = equipment["face_velocity"].str.replace("m/s", "")
equipment["face_velocity_m/s"] = pd.to_numeric(equipment["face_velocity_m/s"], errors="raise")

# For entries "Zentrum", "Irchel", "Unknown", missing, or in m3/h, set face_velocity_m/s to 0.2
equipment.loc[
    (equipment["equipment"] == "fc") &
    (equipment["face_velocity"].isin(["Zentrum", "Irchel", "Unknown"]) 
     | equipment["face_velocity"].isna() 
     | equipment["face_velocity_unit_m3/h"] == True),
    "face_velocity_m/s"
] = 0.2

['280m3/h' '1m/s' nan 'Unknown' '360m3/h' '450m3/h' '325m3/h' '0.5m/s'
 '0.385m/s' '0.397m/s' '400m3/h' '480m3/h' '600m3/h' '0.13m/s' '233m3/h'
 '235m3/h' '260m3/h' '0.35m/s' '340m3/h' 'Irchel' '0.118m/s' '0.4m/s'
 '0.45m/s' '0.6m/s' '396m3/h' '265m3/h' '470m3/h' 'Zentrum']


In [15]:
# Contents lifted

# Replace any "I don't know" or missing values in lifted with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="lifted",
    equipment_value="fc",
    unknown_value="I don't know"
)

In [16]:
# Surface

# Replace any "Unknown" or missing values in surface with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="surface",
    equipment_value="fc"
)

### (1c) Fridge variables (size_fridge)

In [17]:
# Size fridge

# Check unique values for size_fridge and distribution 
print(equipment[equipment["equipment"] == "fridge"]["size_fridge"].unique())

# Before splitting up, replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="size_fridge",
    equipment_value="fridge"
)

# Create variable size_fridge_1 which is "Fan"/"Convection"
equipment.loc[
    (equipment["equipment"] == "fridge") &
    (equipment["size_fridge"].str.contains("fan", case=False, na=False)),
    "size_fridge_1"
] = "Fan"
equipment.loc[
    (equipment["equipment"] == "fridge") &
    (equipment["size_fridge"].str.contains("convection", case=False, na=False)),
    "size_fridge_1"
] = "Convection"

# Create variable size_fridge_2 which extracts the size range (e.g. 250-425L) from size_fridge, and fix the range 250-425 to 250-450
equipment.loc[
    (equipment["equipment"] == "fridge")
    & (equipment["size_fridge"].str.contains("\\(~", na=False)),
    "size_fridge_2"
] = equipment["size_fridge"].str.split("\\(~").str[1].str.split("\\)").str[0].replace({"250-425L": "250-450L"})

#Note fridge has no door_opening option on website, but it still is in the code and was there previously

['Under bench fan assisted (~100-160L)'
 'Under bench convection (~100-160L)'
 'Tall single door fan assisted (~250-425L)'
 'Tall single door fan assisted (~500-700L)'
 'Tall single door convection (~250-425L)'
 'Under bench fan assisted (~250-425L)'
 'Tall double door fan assisted (~1200-1500L)'
 'Tall single door convection (~100-160L)']


### (1d) Freezer variables (size_freezer, temp_freezer, icing, refrigerant)

In [18]:
# Freezer size

# Check unique values for size_freezer and distribution
print(equipment[equipment["equipment"] == "freezer"]["size_freezer"].unique())
print(equipment[equipment["equipment"] == "freezer"]["size_freezer"].value_counts(dropna=False))

# Before splitting up, replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="size_freezer",
    equipment_value="freezer"
)

# Create variable size_freezer_1 which is Chest/Under Bench/Upright
equipment["size_freezer_1"] = equipment["size_freezer"]
equipment["size_freezer_1"] = equipment["size_freezer_1"].replace({
    "Tall and slim single door (~190-38L)": "Upright",
    "Tall and wide single door (~390-525L)": "Upright",
    "Chest (~390-525L)": "Chest",
    "Chest (~190-380L)": "Chest",
    "Under bench (~80-130L)": "Under Bench"
})

# Create variable size_freezer_2 which extracts the size range from size_freezer, and fixes the range 190-38 to 190-380, and 80-130 to 90-130
equipment.loc[
    (equipment["equipment"] == "freezer")
    & (equipment["size_freezer"].str.contains("\\(~", na=False)),
    "size_freezer_2"
] = equipment["size_freezer"].str.split("\\(~").str[1].str.split("\\)").str[0].replace(
    {"190-38L": "190-380L", "80-130L": "90-130L"}
)

print(equipment[equipment["equipment"] == "freezer"]["size_freezer_1"].unique())
print(equipment[equipment["equipment"] == "freezer"]["size_freezer_2"].unique())

['Tall and slim single door (~190-38L)' 'Under bench (~80-130L)'
 'Tall and wide single door (~390-525L)' 'Chest (~390-525L)'
 'Chest (~190-380L)' nan]
size_freezer
Tall and slim single door (~190-38L)     117
Under bench (~80-130L)                   103
Tall and wide single door (~390-525L)     46
Chest (~390-525L)                         15
Chest (~190-380L)                         12
NaN                                        2
Name: count, dtype: int64
['Upright' 'Under Bench' 'Chest']
['190-380L' '90-130L' '390-525L']


In [19]:
# Freezer temp

# Check unique values
print(equipment[equipment["equipment"] == "freezer"]["temp_freezer"].unique())

# Replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="temp_freezer",
    equipment_value="freezer"
)

# Convert to numeric
equipment["temp_freezer"] = pd.to_numeric(equipment["temp_freezer"], errors="raise")

# Bound freezer temperatures to -30 to -20 (anything below -30 becomes -30, anything above -20 becomes -20)
equipment.loc[
    (equipment["equipment"] == "freezer") & (equipment["temp_freezer"] < -30),
    "temp_freezer"
] = -30
equipment.loc[
    (equipment["equipment"] == "freezer") & (equipment["temp_freezer"] > -20),
    "temp_freezer"
] = -20

['-20' '-21' '-26' '-30' '-22' '-25' '-32' '-24' '-20.0' '-18' '-28'
 'Unknown' nan]


In [20]:
# Icing

# Check unique values for icing and distribution
print(equipment[equipment["equipment"] == "freezer"]["icing"].unique())
print(equipment[equipment["equipment"] == "freezer"]["icing"].value_counts(dropna=False))

# Combine "No" and "A little bit" into "No, clear of ice", and "Yes" into "Yes, iced up"
equipment["icing"] = equipment["icing"].replace({
    "No icing": "No, clear of ice",
    "A little building on the drawers and shelves": "No, clear of ice",
    "Yes it's ≥2mm thick and can make opening the door/accessing contents challenging": "Yes, iced up"
})

# Replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="icing",
    equipment_value="freezer"
)

['No icing' 'A little building on the drawers and shelves'
 "Yes it's ≥2mm thick and can make opening the door/accessing contents challenging"
 nan]
icing
A little building on the drawers and shelves                                        148
No icing                                                                            109
Yes it's ≥2mm thick and can make opening the door/accessing contents challenging     36
NaN                                                                                   2
Name: count, dtype: int64


In [21]:
# Refrigerant

print(equipment[equipment["equipment"] == "freezer"]["refrigerant"].unique())
print(equipment[equipment["equipment"] == "freezer"]["refrigerant"].value_counts(dropna=False))

# Replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="refrigerant",
    equipment_value="freezer"
)

['HCs (usually more modern units)' 'HFCs (usually an older unit)' nan
 'Unknown']
refrigerant
HCs (usually more modern units)    192
HFCs (usually an older unit)        90
Unknown                              7
NaN                                  6
Name: count, dtype: int64


In [22]:
# Drawers (Note: Drawer type has switched from Yes+Plastic/NoDrawers/No+Wire to Plastic/Wire)
print(equipment[equipment["equipment"] == "freezer"]["drawers"].unique())
print(equipment[equipment["equipment"] == "freezer"]["drawers"].value_counts(dropna=False))

# Replace "Unknown" or missing with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="drawers",
    equipment_value="freezer"
)

['Yes they are solid plastic' 'Yes they are made from plastic coated wire'
 'No there are no drawers or most are missing' nan]
drawers
Yes they are solid plastic                     242
No there are no drawers or most are missing     30
Yes they are made from plastic coated wire      19
NaN                                              4
Name: count, dtype: int64


In [23]:
# Check whether all freezer variables are non-missing after cleaning
freezer_variables = ["size_freezer_1", "size_freezer_2", "temp_freezer", "icing", "refrigerant", "drawers"]
for var in freezer_variables:
    assert equipment.loc[equipment["equipment"] == "freezer", var].isna().sum() == 0, f"Variable {var} still has missing values for freezer after cleaning."


### (1e) ULT variables (ult_type, temp_ult, size_ult, seals, spacing, filter)

#### ULT type

In [24]:
# Check possible combinations of "Type" and "Age" in equipment_calculations for "Equipment_Type" == "ult_freezer"
print(equipment_calculations.loc[equipment_calculations["Equipment_Type"] == "ult_freezer", ["Type", "Age"]].drop_duplicates())

                    Type           Age
147      Stirling Engine    <10yrs, HC
150  Cascade compressors    <10yrs, HC
159  Cascade compressors    >10yrs, HC
165  Cascade compressors  Any Age, HFC
174     Dual compressors   <10yrs, HFC
180     Dual compressors   >10yrs, HFC
186     Dual compressors   Any Age, HC


In [25]:
# check possible combos of "Type", "Age", "Size" in equipment_calculations for "Equipment_Type" == "ult_freezer"
print(equipment_calculations.loc[equipment_calculations["Equipment_Type"] == "ult_freezer", ["Type", "Age", "Size"]].drop_duplicates())

                    Type           Age           Size
147      Stirling Engine    <10yrs, HC  700-800L Tall
150  Cascade compressors    <10yrs, HC  500-600L Tall
153  Cascade compressors    <10yrs, HC  700-800L Tall
156  Cascade compressors    <10yrs, HC  800-900L Tall
159  Cascade compressors    >10yrs, HC  500-600L Tall
162  Cascade compressors    >10yrs, HC  700-800L Tall
165  Cascade compressors  Any Age, HFC  500-600L Tall
168  Cascade compressors  Any Age, HFC  700-800L Tall
171  Cascade compressors  Any Age, HFC  800-900L Tall
174     Dual compressors   <10yrs, HFC  500-600L Tall
177     Dual compressors   <10yrs, HFC  700-800L Tall
180     Dual compressors   >10yrs, HFC  500-600L Tall
183     Dual compressors   >10yrs, HFC  700-800L Tall
186     Dual compressors   Any Age, HC  700-800L Tall
189     Dual compressors   Any Age, HC  800-900L Tall


In [26]:
# Check unique value for ult_type
print(equipment[equipment["equipment"] == "ult"]["ult_type"].unique())
#print(equipment[equipment["equipment"] == "ult"]["ult_type"].value_counts(dropna=False)) # commented out to reduce output

['Unknown' 'Dual compressor with hydrocarbon refrigerants'
 'Cascade compressors with hydrocarbon refrigerants, under 10 years old'
 'Stirling engine with hydrocarbon refrigerants'
 'Dual compressor with hydrofluorocarbon refrigerants, under 10 years old'
 'Dual compressor with hydrofluorocarbon refrigerants, over 10 years old'
 nan 'Unknown with hydrofluorocarbon refrigerants'
 'Cascade compressors with hydrocarbon refrigerants, over 10 years old'
 'Unknown with hydrocarbon refrigerants, under 10 years old'
 'Cascade compressors with hydrofluorocarbon refrigerants']


In [27]:
# ULT type
# Map survey ult_type to:
# - ult_type_1: engine family (Stirling Engine, Cascade compressors, Dual compressors)
# - ult_type_2: age & refrigerant (<10yrs, HC / >10yrs, HC / Any Age, HC / <10yrs, HFC / Any Age, HFC)
#
# Assumptions for ambiguous/unknown responses:
#   Unknown + HFC  → Dual compressors,    <10yrs, HFC   (mode among HFC responses)
#   Unknown + HC   → Cascade compressors, <10yrs, HC    (mode among HC responses)
#   Unknown / NaN  → Cascade compressors, <10yrs, HC    (overall mode)

ULT_TYPE_MAP = {
    'Stirling engine with hydrocarbon refrigerants':                            ('Stirling Engine',     '<10yrs, HC'),
    'Cascade compressors with hydrocarbon refrigerants, under 10 years old':    ('Cascade compressors', '<10yrs, HC'),
    'Cascade compressors with hydrocarbon refrigerants, over 10 years old':     ('Cascade compressors', '>10yrs, HC'),
    'Cascade compressors with hydrofluorocarbon refrigerants':                  ('Cascade compressors', 'Any Age, HFC'),
    'Dual compressor with hydrocarbon refrigerants':                            ('Dual compressors',    'Any Age, HC'),
    'Dual compressor with hydrofluorocarbon refrigerants, under 10 years old':  ('Dual compressors',    '<10yrs, HFC'),
    'Dual compressor with hydrofluorocarbon refrigerants, over 10 years old':   ('Dual compressors',    '>10yrs, HFC'),
    'Unknown with hydrofluorocarbon refrigerants':                              ('Dual compressors',    '<10yrs, HFC'),
    'Unknown with hydrocarbon refrigerants, under 10 years old':                ('Cascade compressors', '<10yrs, HC'),
    'Unknown':                                                                  ('Cascade compressors', '<10yrs, HC'),
}
ULT_DEFAULT = ('Cascade compressors', '<10yrs, HC')  # for NaN / missing

ult_mask = equipment['equipment'] == 'ult'

mapped = equipment.loc[ult_mask, 'ult_type'].map(
    lambda v: ULT_TYPE_MAP.get(str(v).strip() if pd.notna(v) else '', ULT_DEFAULT)
)
equipment.loc[ult_mask, 'ult_type_1'] = mapped.map(lambda x: x[0]).values
equipment.loc[ult_mask, 'ult_type_2'] = mapped.map(lambda x: x[1]).values

assert equipment.loc[ult_mask, 'ult_type_1'].isna().sum() == 0, "ult_type_1 has missing values"
assert equipment.loc[ult_mask, 'ult_type_2'].isna().sum() == 0, "ult_type_2 has missing values"

print(equipment.loc[ult_mask, ['ult_type', 'ult_type_1', 'ult_type_2']].drop_duplicates().sort_values('ult_type_1').to_string())

                                                                     ult_type           ult_type_1    ult_type_2
28                                                                    Unknown  Cascade compressors    <10yrs, HC
110     Cascade compressors with hydrocarbon refrigerants, under 10 years old  Cascade compressors    <10yrs, HC
515                                                                       NaN  Cascade compressors    <10yrs, HC
625      Cascade compressors with hydrocarbon refrigerants, over 10 years old  Cascade compressors    >10yrs, HC
894                 Unknown with hydrocarbon refrigerants, under 10 years old  Cascade compressors    <10yrs, HC
1488                  Cascade compressors with hydrofluorocarbon refrigerants  Cascade compressors  Any Age, HFC
60                              Dual compressor with hydrocarbon refrigerants     Dual compressors   Any Age, HC
428   Dual compressor with hydrofluorocarbon refrigerants, under 10 years old     Dual compresso

In [28]:
# Temp

# Check unique values for temp_ult
print(equipment[equipment["equipment"] == "ult"]["temp_ult"].unique())
print(equipment[equipment["equipment"] == "ult"]["temp_ult"].value_counts(dropna=False))

#Run set_temp function to move all points to nearest valid option
equipment["temp_ult"] = equipment["temp_ult"].apply(
    lambda t: assign_set_temp(t, [-70, -75, -80])
)
print(equipment[equipment["equipment"] == "ult"]["temp_ult"].unique())

[-70. -80. -75. -78. -82. -81.]
temp_ult
-80.0    86
-70.0    24
-75.0     4
-82.0     2
-81.0     2
-78.0     1
Name: count, dtype: int64
[-70. -80. -75.]


In [29]:
#Size of ULT 

# Check unique values in equipment calculations
print(equipment_calculations.loc[equipment_calculations["Equipment_Type"] == "ult_freezer", ["Size"]].drop_duplicates())

# Check unique values and value counts for size_ult
print(equipment[equipment["equipment"] == "ult"]["size_ult"].unique())
print(equipment[equipment["equipment"] == "ult"]["size_ult"].value_counts(dropna = False))

# Replace unknown or missing with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="size_ult",
    equipment_value="ult"
)

#Must fold Small single door (<500L) into smallest category

equipment["size_ult"] = np.where(
    equipment["size_ult"].str.contains("500", na=False), "500-600L Tall", #will cover both categories
    np.where(equipment["size_ult"].str.contains("700", na=False), "700-800L Tall",
             np.where(equipment["size_ult"].str.contains("900", na=False), "800-900L Tall",
                      equipment["size_ult"]))
)

print(equipment[equipment["equipment"] == "ult"]["size_ult"].value_counts(dropna = False))

              Size
147  700-800L Tall
150  500-600L Tall
156  800-900L Tall
['Tall single door (~700-800L)' 'Tall single door (~500-600L)'
 'Small single door (<500L)' 'Tall single door (~800-900L)' 'Unknown']
size_ult
Tall single door (~700-800L)    60
Tall single door (~500-600L)    36
Tall single door (~800-900L)    13
Small single door (<500L)        8
Unknown                          2
Name: count, dtype: int64
size_ult
700-800L Tall    62
500-600L Tall    44
800-900L Tall    13
Name: count, dtype: int64


In [30]:
# Seals

# Check unique values for seals and distribution
print(equipment[equipment["equipment"] == "ult"]["seals"].unique())
print(equipment[equipment["equipment"] == "ult"]["seals"].value_counts(dropna = False))

# No "Unknown" or missing --> no further cleaning needed

# Create damaged seals penalty indicator variable (1 if seals are damaged, 0 if seals are intact)
equipment.loc[
    (equipment["equipment"] == "ult")
    & (equipment["seals"] == "Clean and in good condition"), 
    "damaged_seals_penalty"] = 0
equipment.loc[
    (equipment["equipment"] == "ult") 
    & (equipment["seals"].isin(["Dirty or lightly iced", "Damaged, absent in places, or with a layer of ice"])), 
    "damaged_seals_penalty"] = 1

print(equipment[equipment["equipment"] == "ult"][["seals", "damaged_seals_penalty"]].value_counts(dropna = False))

['Clean and in good condition' 'Dirty or lightly iced'
 'Damaged, absent in places, or with a layer of ice']
seals
Clean and in good condition                          80
Dirty or lightly iced                                36
Damaged, absent in places, or with a layer of ice     3
Name: count, dtype: int64
seals                                              damaged_seals_penalty
Clean and in good condition                        0.0                      80
Dirty or lightly iced                              1.0                      36
Damaged, absent in places, or with a layer of ice  1.0                       3
Name: count, dtype: int64


In [31]:
# Spacing

# Check unique values for spacing and distribution
print(equipment[equipment["equipment"] == "ult"]["spacing"].unique())
print(equipment[equipment["equipment"] == "ult"]["spacing"].value_counts(dropna=False))

# No "Unknown" or missing --> no further cleaning needed

# Create no spacing penalty indicator variable
equipment.loc[
    (equipment["equipment"] == "ult")
    & (equipment["spacing"] == "There's little space around the unit and we store items on top of the unit."),
    "no_spacing_penalty"] = 1
equipment.loc[
    (equipment["equipment"] == "ult")
    & (equipment["spacing"] == "≥150mm gap at the back and sides of the unit, with no items stored on top."),
    "no_spacing_penalty"] = 0

print(equipment[equipment["equipment"] == "ult"][["spacing", "no_spacing_penalty"]].value_counts(dropna = False))

["There's little space around the unit and we store items on top of the unit."
 '≥150mm gap at the back and sides of the unit, with no items stored on top.']
spacing
≥150mm gap at the back and sides of the unit, with no items stored on top.     102
There's little space around the unit and we store items on top of the unit.     17
Name: count, dtype: int64
spacing                                                                      no_spacing_penalty
≥150mm gap at the back and sides of the unit, with no items stored on top.   0.0                   102
There's little space around the unit and we store items on top of the unit.  1.0                    17
Name: count, dtype: int64


In [32]:
# Filter

# Check unique values 
print(equipment[equipment["equipment"] == "ult"]["filter"].unique())
print(equipment[equipment["equipment"] == "ult"]["filter"].value_counts(dropna=False))

# Fill missing/unknown with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="filter",
    equipment_value="ult"
)

# Create clogged filter penalty indicator variable (1 if clogged or a little dirty, 0 if clear or no filter - check this)
equipment.loc[
    (equipment["equipment"] == "ult") &
    (equipment["filter"].isin(["Most of it is clogged", "It's a little dirty"])),
    "clogged_filter_penalty"] = 1
equipment.loc[
    (equipment["equipment"] == "ult") &
    (equipment["filter"].isin(["No filter", "It's clear"])),
    "clogged_filter_penalty"] = 0

["It's clear" 'Most of it is clogged' "It's a little dirty" 'Unknown'
 'No filter' nan]
filter
It's clear               70
It's a little dirty      42
Most of it is clogged     2
No filter                 2
NaN                       2
Unknown                   1
Name: count, dtype: int64


In [33]:
# Check whether all ult variables are non-missing after cleaning
ult_variables = ["ult_type_1", "ult_type_2", "temp_ult", "size_ult", "no_spacing_penalty", "damaged_seals_penalty", "clogged_filter_penalty"]
for var in ult_variables:
    assert equipment.loc[equipment["equipment"] == "ult", var].isna().sum() == 0, f"Variable {var} still has missing values for ult after cleaning."


### (1f) Glassware variables

In [34]:
# Capacity

# Check unique values for Size in equipment_calculations for "Equipment_Type" == "drying_cabinet"
print(equipment_calculations.loc[equipment_calculations["Equipment_Type"] == "drying_cabinet", "Size"].drop_duplicates())

#Capacity reformating, just dropping first character
print(equipment[equipment["equipment"] == "glassware"]["capacity_glassware"].unique())
equipment["capacity_glassware"] = equipment["capacity_glassware"].str.split("~").str[1]
print(equipment[equipment["equipment"] == "glassware"]["capacity_glassware"].unique())

64      90-100L
70     180-200L
76     425-550L
82    850-1000L
Name: Size, dtype: object
['~90-100L' '~850-1000L' '~180-200L' '~425-550L']
['90-100L' '850-1000L' '180-200L' '425-550L']


In [35]:
# Tech

#tech is now age and has simpler formatting
print(equipment[equipment["equipment"] == "glassware"]["tech"].unique())
equipment["tech"] = np.where(
    equipment["tech"].str.contains("modern", na=False), "Newer", 
    np.where(equipment["tech"].str.contains("older", na=False), "Older", 
             equipment["tech"])
)
print(equipment[equipment["equipment"] == "glassware"]["tech"].unique())

["The chamber is insulated, it's warm to the touch when on (modern technology)"
 "The chamber has no insulation, it's hot to touch when on (older technology)"]
['Newer' 'Older']


In [36]:
# Fan
#Map to correct options that match wording
print(equipment[equipment["equipment"] == "glassware"]["fan"].value_counts(dropna=False))

# Replace unknown or missing with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="fan",
    equipment_value="glassware",
    unknown_value="I don't know"
)

# Create fan penalty variable (1 if fan, 0 if no fan)
equipment.loc[
    (equipment["equipment"] == "glassware") &
    (equipment["fan"] == "Yes"),
    "fan_penalty"] = 1
equipment.loc[
    (equipment["equipment"] == "glassware") &
    (equipment["fan"] == "No"),
    "fan_penalty"] = 0

fan
Yes             26
No              10
I don't know     3
Name: count, dtype: int64


In [37]:
# Temp

# Check unique values for temp_glassware
print(equipment[equipment["equipment"] == "glassware"]["temp_glassware"].unique())
# print(equipment[equipment["equipment"] == "glassware"]["temp_glassware"].value_counts(dropna=False)) # commented out to reduce output

# Replace unknown or missing with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="temp_glassware",
    equipment_value="glassware",
    unknown_value="I don't know"
)

#Assign values to nearest valid temperature
equipment["temp_glassware"] = equipment["temp_glassware"].apply(
    lambda t: assign_set_temp(t, [50, 60, 75])
)

# Check unique values after cleaning
print(equipment[equipment["equipment"] == "glassware"]["temp_glassware"].value_counts(dropna=False))

['60' '65' '50' '80' "I don't know" '85' '51' '75']
temp_glassware
50.0    21
60.0    12
75.0     6
Name: count, dtype: int64


In [38]:
# Check whether all glassware variables are non-missing after cleaning
glassware_variables = ["capacity_glassware", "tech", "fan_penalty", "temp_glassware"]
for var in glassware_variables:
    assert equipment.loc[equipment["equipment"] == "glassware", var].isna().sum() == 0, f"Variable {var} still has missing values for glassware after cleaning."

### (1g) MSC variables

In [39]:
# Width

#Already is a numeric value
print(equipment[equipment["equipment"] == "microbio"]["width"].unique())
print(equipment[equipment["equipment"] == "microbio"]["width"].value_counts(dropna=False))

# Replace unknown or missing with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="width",
    equipment_value="microbio"
)

#Remove invalid option
equipment["width"] = equipment["width"].replace({1600: 1500})

[1500. 1200.  900. 1800.   nan 1600.]
width
1200.0    53
1800.0    29
1500.0    15
900.0     12
NaN        4
1600.0     2
Name: count, dtype: int64


In [40]:
# Age

# Check unique values for age_microbio and distribution
print(equipment[equipment["equipment"] == "microbio"]["age_microbio"].unique())
print(equipment[equipment["equipment"] == "microbio"]["age_microbio"].value_counts(dropna=False))

#Age values must match format
equipment["age_microbio"] = equipment["age_microbio"].replace({"Less than 5 years old": "<5 yrs",
                                                               "5-20 years old": "5-20yrs",
                                                               "More than 20 years old": ">20 yrs"})

#Replace I don't know for age with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="age_microbio",
    equipment_value="microbio",
    unknown_value="I don't know"
)

['Less than 5 years old' '5-20 years old' 'More than 20 years old'
 "I don't know"]
age_microbio
5-20 years old            69
Less than 5 years old     33
More than 20 years old    11
I don't know               2
Name: count, dtype: int64


In [41]:
# Ducting
print(equipment[equipment["equipment"] == "microbio"]["ducting"].unique())
print(equipment[equipment["equipment"] == "microbio"]["ducting"].value_counts(dropna=False))

# Create ducting penalty variable 
equipment.loc[
    (equipment["equipment"] == "microbio") &
    (equipment["ducting"] == "Yes"),
    "ducting_penalty"] = 1
equipment.loc[
    (equipment["equipment"] == "microbio") &
    (equipment["ducting"] == "No"),
    "ducting_penalty"] = 0

['No' 'Yes']
ducting
No     110
Yes      5
Name: count, dtype: int64


### (1h) Incubator variables

In [42]:
# Capacity
print(equipment[equipment["equipment"] == "incubator"]["capacity_incubator"].unique())

#No <150L, Add to 150-170L
equipment["capacity_incubator"] = equipment["capacity_incubator"].replace({"<150L": "150-170L",
                                                                           "~150-170L": "150-170L",
                                                                           "~220-260L": "220-260L"})

['~220-260L' '~150-170L' '<150L']


In [43]:
# Age
print(equipment[equipment["equipment"] == "incubator"]["age_incubator"].unique())

# For age, <=20 years/>20 years -> Newer/Older
equipment["age_incubator"] = equipment["age_incubator"].replace({"≤ 20 years": "Newer",
                                                                  "> 20 years": "Older"})

['≤ 20 years' '> 20 years']


### (1i) Water bath variables

In [44]:
# Capacity
print(equipment[equipment["equipment"] == "bath"]["capacity_bath"].unique())
print(equipment[equipment["equipment"] == "bath"]["capacity_bath"].value_counts(dropna=False))

print(equipment_calculations.loc[equipment_calculations["Equipment_Type"] == "water_bath", "Size"].drop_duplicates())

# Replace missing/unknown with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="capacity_bath",
    equipment_value="bath"
)

# Modify "capacity_bath" to combine "10-12L" and "6-10L" into "10L". Currently mapped based on distance to median
equipment["capacity_bath"] = equipment["capacity_bath"].replace({"10-12L": "10L",
                                                                 "6-10L": "10L",
                                                                 "Less than 5L": "5-6L"})

['10-12L' '5-6L' '12-14L' '18-20L' nan 'Less than 5L' '6-10L']
capacity_bath
5-6L            68
10-12L          41
12-14L          18
18-20L           8
Less than 5L     4
6-10L            2
NaN              1
Name: count, dtype: int64
0      5-6L
1       10L
2    12-14L
3    18-20L
Name: Size, dtype: object


In [45]:
# Temp

print(equipment[equipment["equipment"] == "bath"]["temp_bath"].unique())
print(equipment[equipment["equipment"] == "bath"]["temp_bath"].value_counts(dropna=False))

# Fill missing/Unknown values for capacity_bath, heating, temp_bath, lid
equipment = fill_with_equipment_mode(
    equipment,
    value_col="temp_bath",
    equipment_value="bath",
)

# Assign values to nearest valid temperature (37, 65, 90)
equipment["temp_bath"] = equipment["temp_bath"].apply(
    lambda t: assign_set_temp(t, [37, 65, 90])
)

print(equipment[equipment["equipment"] == "bath"]["temp_bath"].value_counts(dropna=False))

['37' '42.0' 'Unknown' '90' '85' '65' '42' '51' '41' '40' '22' nan '49'
 '46' '56' '58']
temp_bath
37         76
65         11
40         10
Unknown     8
42          8
NaN         5
42.0        4
90          4
85          4
51          2
41          2
49          2
56          2
58          2
22          1
46          1
Name: count, dtype: int64
temp_bath
37.0    119
65.0     15
90.0      8
Name: count, dtype: int64


In [46]:
# Heating
print(equipment[equipment["equipment"] == "bath"]["heating"].unique())

# Fill missing/Unknown values 
equipment = fill_with_equipment_mode(
    equipment,
    value_col="heating",
    equipment_value="bath",
)

# Create beads penalty variable (1 if beads, 0 if water)
equipment.loc[
    (equipment["equipment"] == "bath") &
    (equipment["heating"] == "Metal beads"),
    "beads_penalty"] = 1
equipment.loc[
    (equipment["equipment"] == "bath") &
    (equipment["heating"] == "Water"),
    "beads_penalty"] = 0

['Water' 'Metal beads' nan]


In [47]:
# Lid
print(equipment[equipment["equipment"] == "bath"]["lid"].unique())

# Fill missing/Unknown values
equipment = fill_with_equipment_mode(
    equipment,
    value_col="lid",
    equipment_value="bath",
)

# Create lid penalty variable (1 if lid is "No" (lid off), 0 if lid is "Yes" (lid on))
equipment.loc[
    (equipment["equipment"] == "bath") &
    (equipment["lid"] == "No"),
    "lid_off_penalty"] = 1
equipment.loc[
    (equipment["equipment"] == "bath") &
    (equipment["lid"] == "Yes"),
    "lid_off_penalty"] = 0

['Yes' 'No' nan]


In [48]:
# Check whether all bath variables are non-missing after cleaning
bath_variables = ["capacity_bath", "temp_bath", "beads_penalty", "lid_off_penalty"]
for var in bath_variables:
    assert equipment.loc[equipment["equipment"] == "bath", var].isna().sum() == 0, f"Variable {var} still has missing values for bath after cleaning."

### (1j) Cryostat variables

In [49]:
# Temp
print(equipment[equipment["equipment"] == "cryostat"]["temp_cryostat"].value_counts(dropna=False))

# Fill missing/Unknown values for temp_cryostat
equipment = fill_with_equipment_mode(
    equipment,
    value_col="temp_cryostat",
    equipment_value="cryostat",
)

# Assign values to nearest valid temperature (-25, -20, -18)
equipment["temp_cryostat"] = equipment["temp_cryostat"].apply(
    lambda t: assign_set_temp(t, [-25, -20, -18]) #ensures only SPARKHub values are in data
)

temp_cryostat
-20        13
Unknown     2
-25         1
Name: count, dtype: int64


In [50]:
# Sleep mode
print(equipment[equipment["equipment"] == "cryostat"]["sleep_mode"].value_counts(dropna=False))

# Fill missing/Unknown values
equipment = fill_with_equipment_mode(
    equipment,
    value_col="sleep_mode",
    equipment_value="cryostat",
)

# Modify sleep_mode variable with "Energy Saving Mode Available"/"No Energy Saving Mode Available" based on sleep_mode
equipment["sleep_mode"] = np.where(
    (equipment["sleep_mode"].isin([
        "Yes, we use the unit 8am to 5pm, outside these hours it's in sleep mode",
        "Yes, but it's not used",
        "Yes, we use the unit 9am to 5pm, outside these hours it's in sleep mode"
        ]) | equipment["sleep_mode"].isna())    
        & (equipment["equipment"] == "cryostat"),
    "Energy Saving Mode Available", #All these values are treated as showing energy-saving mode is available now
    np.where((equipment["sleep_mode"] == "No, it does not")
             & (equipment["equipment"] == "cryostat"),
             "No Energy Saving Mode Available",
             equipment["sleep_mode"])
)

print(equipment[equipment["equipment"] == "cryostat"]["sleep_mode"].value_counts(dropna=False))

sleep_mode
No, it does not                                                            7
Yes, we use the unit 9am to 5pm, outside these hours it's in sleep mode    6
Yes, we use the unit 8am to 5pm, outside these hours it's in sleep mode    2
Yes, but it's not used                                                     1
Name: count, dtype: int64
sleep_mode
Energy Saving Mode Available       9
No Energy Saving Mode Available    7
Name: count, dtype: int64


In [51]:
# Check whether all cryostat variables are non-missing after cleaning
cryostat_variables = ["temp_cryostat", "sleep_mode"]
for var in cryostat_variables:
    assert equipment.loc[equipment["equipment"] == "cryostat", var].isna().sum() == 0, f"Variable {var} still has missing values for cryostat after cleaning."

### (1k) Block heater variables

In [52]:
# Block
print(equipment[equipment["equipment"] == "heater"]["blocks"].unique())
print(equipment[equipment["equipment"] == "heater"]["blocks"].value_counts(dropna=False))

# Fill missing/Unknown values for blocks
equipment = fill_with_equipment_mode(
    equipment,
    value_col="blocks",
    equipment_value="heater",
)

#Match format, currently includes word "block"
equipment["blocks"]= equipment["blocks"].str.extract(r'^(\d+)') #Extracts the numeric characters

['1-Block' '3-Block' '2-Block' '4-Block' nan]
blocks
1-Block    89
2-Block    30
3-Block     8
4-Block     6
NaN         4
Name: count, dtype: int64


In [53]:
# Temp
print(equipment_calculations.loc[equipment_calculations["Equipment_Type"] == "heat_block", "Set_Temp"].drop_duplicates())

print(equipment[equipment["equipment"] == "heater"]["temp_heater"].unique())
# print(equipment[equipment["equipment"] == "heater"]["temp_heater"].value_counts(dropna=False)) # commented out to reduce output

# Replace missing/Unknown values for temp_heater
equipment = fill_with_equipment_mode(
    equipment,
    value_col="temp_heater",
    equipment_value="heater",
)

# Assign values to nearest valid temperature (37, 65, 95, 100)
equipment["temp_heater"] = equipment["temp_heater"].apply(
    lambda t: assign_set_temp(t, [37, 65, 95, 100]) #ensures only SPARKHub values are in data
)

print(equipment[equipment["equipment"] == "heater"]["temp_heater"].value_counts(dropna=False))

48     37.0
52     65.0
56     95.0
60    100.0
Name: Set_Temp, dtype: float64
['95' '37' 'Unknown' '65' '100' '120' '60' '61' '51' '50' '90' '66' '45'
 '76' '10' nan]
temp_heater
65.0     82
37.0     28
95.0     22
100.0     5
Name: count, dtype: int64


In [54]:
# Check whether all heater variables are non-missing after cleaning
heater_variables = ["blocks", "temp_heater"]
for var in heater_variables:
    assert equipment.loc[equipment["equipment"] == "heater", var].isna().sum() == 0, f"Variable {var} still has missing values for heater after cleaning."

### (1l) IT variables

In [55]:
# IT type
print(equipment[equipment["equipment"] == "it"]["it_type"].unique())
print(equipment[equipment["equipment"] == "it"]["it_type"].value_counts(dropna=False))

#Fix cases
equipment["it_type"] = equipment["it_type"].replace(
    {'High-powered desktop computer with graphics card': 'High-Powered Desktop Computer with Graphics Card', 
     'Standard desktop computer': 'Standard Desktop Computer',
     'Standard laptop': 'Standard Laptop',
     'High-powered laptop with graphics card': 'High-Powered Laptop with Graphics Card'}
)

['High-powered desktop computer with graphics card' 'Standard laptop'
 'Standard desktop computer' 'High-powered laptop with graphics card'
 'Screen only']
it_type
Standard laptop                                     250
Standard desktop computer                           231
High-powered desktop computer with graphics card     79
High-powered laptop with graphics card               17
Screen only                                           4
Name: count, dtype: int64


In [56]:
# Monitor
print(equipment[equipment["equipment"] == "it"]["monitor"].unique())
# print(equipment[equipment["equipment"] == "it"]["monitor"].value_counts(dropna=False)) # comment out to reduce output

# If missing, set to "It does not use an external monitor"
equipment.loc[
    (equipment["equipment"] == "it") &
    (equipment["monitor"].isna()),
    "monitor"
] = "It does not use an external monitor"

# Monitor variable split for size (with the other equipments) and brightness (its own column)
#Extract size related information and match to accepted values into a new column
equipment["monitor_size"] = np.where(
    equipment["monitor"].str.contains("Small", na=False), "19-22 inches",
    np.where(equipment["monitor"].str.contains("Medium", na=False), "24-27 inches",
             np.where(equipment["monitor"].str.contains("Large", na=False), "30+ inches",
                      np.where(equipment["monitor"].str.contains("does not", na=False), "No monitor", #to allow for the actual computers
                               equipment["monitor"]))) #Set to monitor value
)
#Extract brightness related information and match to accepted values into a new column
equipment["monitor_brightness"] = np.where(
    equipment["monitor"].str.contains("lowest", na=False), "Lowest",
    np.where(equipment["monitor"].str.contains("mid", na=False), "Mid",
             np.where(equipment["monitor"].str.contains("full", na=False), "Full",
                      equipment["monitor"])) #Set to monitor value
)

# Check unique values after splitting
print(equipment[equipment["equipment"] == "it"][["monitor", "monitor_brightness", "monitor_size"]].value_counts(dropna=False))

['Medium (24-27 inches), mid brightness'
 'Large (30+ inches), mid brightness'
 'Small (19-22 inches), mid brightness'
 'Medium (24-27 inches), lowest brightness'
 'It does not use an external monitor'
 'Medium (24-27 inches), full brightness' nan
 'Small (19-22 inches), full brightness'
 'Small (19-22 inches), lowest brightness'
 'Large (30+ inches), full brightness'
 'Large (30+ inches), lowest brightness']
monitor                                   monitor_brightness                   monitor_size
Medium (24-27 inches), mid brightness     Mid                                  24-27 inches    336
Large (30+ inches), mid brightness        Mid                                  30+ inches       63
It does not use an external monitor       It does not use an external monitor  No monitor       59
Small (19-22 inches), mid brightness      Mid                                  19-22 inches     55
Medium (24-27 inches), full brightness    Full                                 24-27 inches     29


### (1m) Combine and rename variables to prepare for merging

In [57]:
# Create combined "type" variable
# Type = [controller_type, first half size_fridge, first half size_freezer, first half of ult_type, sleep_mode, it_type]
def equipment_type_func(row):
    if row["equipment"] == "fc":
        return row["controller_type"]
    elif row["equipment"] == "fridge":
        return row["size_fridge_1"]
    elif row["equipment"] == "freezer":
        return row["size_freezer_1"]
    elif row["equipment"] == "ult":
        return row["ult_type_1"]
    elif row["equipment"] == "cryostat":
        return row["sleep_mode"]
    elif row["equipment"] == "it":
        return row["it_type"]
    else:
        return None
equipment["type"] = equipment.apply(lambda row: equipment_type_func(row), axis=1)


In [58]:
# Create combined "size" variable
# Size = [second half size_fridge, second half size_freezer, second half size_ult, capacity_bath, capacity_incubator, first half monitor]
def equipment_size_func(row):
    if row["equipment"] == "fridge":
        return row["size_fridge_2"]
    elif row["equipment"] == "freezer":
        return row["size_freezer_2"]
    elif row["equipment"] == "ult":
        return row["size_ult"]
    elif row["equipment"] == "glassware":
        return row["capacity_glassware"]
    elif row["equipment"] == "bath":
        return row["capacity_bath"]
    elif row["equipment"] == "incubator":
        return row["capacity_incubator"]
    elif row["equipment"] == "it":
        return row["monitor_size"]
    else:
        return None
equipment["size"] = equipment.apply(lambda row: equipment_size_func(row), axis=1)

In [59]:
# Create combined "age" variable
# Age = [second half of ult_type (with age), tech, age_microbio, age_incubator]
def equipment_age_func(row):
    if row["equipment"] == "ult":
        return row["ult_type_2"]
    if row["equipment"] == "glassware":
        return row["tech"]
    elif row["equipment"] == "cryostat":
        return row["tech"]
    elif row["equipment"] == "microbio":
        return row["age_microbio"]
    elif row["equipment"] == "incubator":
        return row["age_incubator"]
    else:
        return None
equipment["age"] = equipment.apply(lambda row: equipment_age_func(row), axis=1)


In [60]:
# Create combined "temp" variable
# set_temp = [temp_freezer, temp_ult, temp_glassware, temp_cryostat, temp_bath, temp_heater]
def equipment_temp_func(row):
    if row["equipment"] == "freezer":
        return row["temp_freezer"]
    elif row["equipment"] == "ult":
        return row["temp_ult"]
    elif row["equipment"] == "glassware":
        return row["temp_glassware"]
    elif row["equipment"] == "cryostat":
        return row["temp_cryostat"]
    elif row["equipment"] == "bath":
        return row["temp_bath"]
    elif row["equipment"] == "heater":
        return row["temp_heater"]
    else:
        return None
equipment["set_temp"] = equipment.apply(lambda row: equipment_temp_func(row), axis=1)
equipment["set_temp"] = pd.to_numeric(equipment["set_temp"], errors='coerce')


In [61]:

# 3. Create columns to match and force variable types
## Process which shifts all the relevant data to the same column name that Isobel used.
## Changes regarding correct exact entries to be made afterwards
## Each section has a function which ensures only the relevant equipment have their values from these columns selected
## Other equipment gets None as the value


# Penalty indicators = [icing, drawers, seals, spacing, filter, fan, ducting, heating, lid, refrigerant]

# fc_specific = [face_velocity, lifted, surface, ] TODO look at code for this information

In [62]:
# # print(equipment.columns.values)
# # print(equipment.shape)
# print(equipment.iloc[:9, 57:64].dtypes)
# #Examine each new column individually for data verification
# def column_checker(colname):
#     print("Counts for " + colname + "\n")
#     print(equipment[colname].dtype.name)
#     print(equipment[colname].unique())

# column_checker("Type")
# #We have no computer screen types? -> First merge on IT type, then on on the size/brightness etc
# column_checker("Work_Surface_Width")
# column_checker("Size")
# column_checker("N_blocks")
# column_checker("Age")
# column_checker("Set_Temp")
# column_checker("Brightness")
# # print(equipment["Type"])


## Clean up and save processed data

In [63]:
# Save processed data
equipment.to_csv(config.PROCESSED_DATA / "panel_processed_3.csv", index =False)